# 10 · End-to-end Case Studies

The complete chain for individual products: data → forecast → risk → recommendation → explanation → what-if.

> **Data provenance.** Sales, dates, prices, SKUs and countries are **real**
> (UCI Online Retail II, CC BY 4.0, 1,067,371 invoice lines).
> Stock levels, suppliers, lead times, MOQ, pack sizes and every cost are
> **SIMULATED** — the source dataset contains none of them. See `sim/` and
> `docs/pbl1_scope.md`.


In [1]:
import sys, os
from pathlib import Path
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from xaierp.utils.config import load_config, resolve
CFG = load_config()
REPORTS = resolve(CFG["paths"]["reports"])
FIGS = resolve(CFG["paths"]["figures"])
print("project root:", ROOT)


project root: E:\downloads\ML Project


## The six questions this project set out to answer

In [2]:
rec = pd.read_parquet(resolve(CFG['paths']['results']) / 'procurement_recommendations.parquet')
s = json.loads((REPORTS / 'model_summary.json').read_text())
print('Test period:', s['test_date_min'], '->', s['test_date_max'])
print(f"{len(rec):,} decisions across {rec['stock_code'].nunique()} SKUs")

Test period: 2011-08-24 -> 2011-12-02
16,689 decisions across 198 SKUs


In [3]:
from xaierp.explain.decision_explanation import explain_decision, render_explanation
from xaierp.explain.whatif import run_whatif_table
from xaierp.policy.reorder import decide_reorder, safety_stock_from_quantiles

def full_story(row, review=7):
    lead = int(row['lead_time_days'])
    scale = (lead + review) / CFG['target']['horizon_days']
    q50, q95 = max(0.0, row['pred_q50']), max(0.0, row['pred_q95'])
    f7 = float(row['pred_best_regression'])
    d = decide_reorder(
        expected_demand_leadtime=max(0.0, f7) * scale,
        safety_stock=safety_stock_from_quantiles(q95, q50, scale=scale),
        inventory_position=float(row['inventory_position']),
        lead_time_days=lead, moq=int(row['moq']),
        pack_size=int(row['pack_size']),
        supplier_capacity=int(row['supplier_capacity']),
        unit_cost=float(row['unit_cost']),
        mean_daily_demand=float(row['rolling_mean_28']))
    exp = explain_decision(str(row['stock_code']), str(row['description']),
                           row['date'], d, f7,
                           float(row['prob_best_classifier']), None,
                           float(row['rolling_mean_28']))
    return d, exp

## Case A — a high-risk, stocked-out product

In [4]:
row = rec.loc[rec['prob_best_classifier'].idxmax()]
d, exp = full_story(row)
print(render_explanation(exp))
print(f"\nActual demand over the next 7 days (hindsight): "
      f"{row['future_7d_demand']:,.0f} units")

Product            : 22423  REGENCY CAKESTAND 3 TIER
Decision date      : 2011-11-30
Current stock      : 311 units
Predicted 7-day demand : 390 units
Supplier lead time : 8 days
Stockout risk      : HIGH  (p = 1.00)
Recommendation     : REORDER
Recommended order  : 888 units
Order value        : 6,420.68  (SIMULATED cost basis)

Why is demand forecast at this level?
  (no attribution available for this row)

Why this order quantity?
  1. Expected demand over the 8-day lead time plus the review period is 836 units.
  2. A safety buffer of 361 units covers forecast error at the target service level.
  3. 311 units are already available or on order, which reduces the requirement.
  4. Rounding up to whole packs of 24 adds 2.2 units.

Quantity decomposition (sums to the recommendation):
       +835.59   expected demand over lead time
       +361.37   safety stock
       -311.13   less: stock already available / on order
         +0.00   clip at zero (never order negative)
         +0.00  

## Case B — a well-stocked product that needs nothing

In [5]:
row = rec.loc[rec['prob_best_classifier'].idxmin()]
d, exp = full_story(row)
print(render_explanation(exp))
print(f"\nActual demand over the next 7 days (hindsight): "
      f"{row['future_7d_demand']:,.0f} units")

Product            : 48173C  DOORMAT BLACK FLOCK 
Decision date      : 2011-09-08
Current stock      : 618 units
Predicted 7-day demand : 67 units
Supplier lead time : 6 days
Stockout risk      : LOW  (p = 0.00)
Recommendation     : DO NOT REORDER
Recommended order  : 0 units
Order value        : 0.00  (SIMULATED cost basis)

Why is demand forecast at this level?
  (no attribution available for this row)

Why this order quantity?
  1. Expected demand over the 6-day lead time plus the review period is 124 units.
  2. A safety buffer of 111 units covers forecast error at the target service level.
  3. 618 units are already available or on order, which reduces the requirement.
  4. Stock already exceeds the requirement, so the raw calculation was negative and has been clipped to zero.

Quantity decomposition (sums to the recommendation):
       +123.63   expected demand over lead time
       +110.74   safety stock
       -617.93   less: stock already available / on order
       +383.56   

## What-if on the high-risk case

In [6]:
row = rec.loc[rec['prob_best_classifier'].idxmax()]
base = {'demand_7d': float(row['pred_best_regression']),
        'q50': max(0.0, float(row['pred_q50'])),
        'q95': max(0.0, float(row['pred_q95'])),
        'current_stock': float(row['inventory_position']),
        'lead_time_days': int(row['lead_time_days']),
        'review_period': 7, 'horizon': CFG['target']['horizon_days'],
        'moq': int(row['moq']), 'pack_size': int(row['pack_size']),
        'supplier_capacity': int(row['supplier_capacity']),
        'unit_cost': float(row['unit_cost']),
        'mean_daily_demand': float(row['rolling_mean_28'])}
pd.DataFrame(run_whatif_table(base))

,scenario,lead_time_days,inventory_position,expected_demand,safety_stock,moq,recommended_qty,change_vs_current,order_value,reorder
0,Current situation,8,311.1,835.6,361.4,12,888,0,6420.68,True
1,Lead time halved (8 -> 4 days),4,311.1,612.8,265.0,12,576,-312,4164.77,True
2,Lead time doubled (8 -> 16 days),16,311.1,1281.2,554.1,12,1536,648,11106.05,True
3,Stock fully depleted (0 units),8,0.0,835.6,361.4,12,1200,312,8676.60,True
4,Stock = 50% of lead-time demand,8,417.8,835.6,361.4,12,792,-96,5726.56,True
5,Stock = 150% of lead-time demand,8,1253.4,835.6,361.4,12,0,-888,0.00,False
6,No safety buffer (service level dropped),8,311.1,835.6,0.0,12,528,-360,3817.70,True
7,Demand surge +30%,8,311.1,1086.3,361.4,12,1152,264,8329.54,True
8,Supplier raises MOQ to 288,8,311.1,835.6,361.4,288,888,0,6420.68,True


Each row re-runs the **real** policy function with one input changed — nothing is interpolated. This is what makes the system decision *support*: a buyer can test a negotiation ('what if I get the lead time down to 4 days?') before committing.

## Aggregate view of the recommendations

In [7]:
latest = rec.sort_values('date').groupby('stock_code').tail(1)
o = latest[latest['rec_reorder']]
print(f'SKUs needing a reorder : {len(o)} of {len(latest)}')
print(f'Total units            : {o["rec_qty"].sum():,.0f}')
print(f'Total value (SIMULATED): {o["rec_order_value"].sum():,.0f}')
o[['stock_code','description','current_stock','pred_best_regression',
   'rec_qty','lead_time_days','rec_order_value']].sort_values(
   'rec_order_value', ascending=False).head(15).round(1)

SKUs needing a reorder : 190 of 198
Total units            : 146,455
Total value (SIMULATED): 212,511


,stock_code,description,current_stock,pred_best_regression,rec_qty,lead_time_days,rec_order_value
16602,85123A,WHITE HANGING HEART T-LIGHT HOLDER,804.7,1286.199951,6024,9,8799.9
11083,22910,PAPER CHAIN KIT VINTAGE CHRISTMAS,0.0,1195.099976,3878,9,5800.3
3476,21623,VINTAGE UNION JACK MEMOBOARD,114.5,106.800003,1044,7,4722.3
15920,84997D,PINK 3 PIECE MINI DOTS CUTLERY SET,0.0,202.699997,2419,11,4154.6
7378,22423,REGENCY CAKESTAND 3 TIER,311.1,367.299988,552,8,3991.2
5501,22114,HOT WATER BOTTLE TEA AND SYMPATHY,89.9,645.599976,1596,9,3480.6
12357,23298,SPOTTY BUNTING,134.7,269.299988,1278,16,3437.6
14558,84029E,RED WOOLLY HOTTIE WHITE HEART.,463.3,386.000000,1740,7,3221.4
13891,79321,CHILLI LIGHTS,498.3,468.600006,1129,3,3091.0
8199,22502,PICNIC BASKET WICKER SMALL,116.5,68.099998,1140,5,3088.3


## Final scorecard

In [8]:
pol = pd.read_csv(REPORTS / 'policy_comparison.csv')
p0 = pol[pol['policy']=='P0'].iloc[0]; p4 = pol[pol['policy']=='P4'].iloc[0]
print(f"Forecast  : {s['best_regression_model']}  "
      f"RMSE {s['best_regression_test_RMSE']:.1f}  R2 {s['best_regression_test_R2']:.3f}")
print(f"Classifier: {s['best_classifier_model']}  "
      f"F1 {s['best_classifier_test_f1']:.3f}  AUC {s['best_classifier_test_roc_auc']:.3f}")
print(f"Fill rate : {p0['fill_rate']:.1%} (P0 baseline) -> {p4['fill_rate']:.1%} (P4)")
print(f"Cost      : {p4['cost_vs_P0_pct']:+.1f}% vs the P0 baseline")

Forecast  : Ridge Regression (L2)  RMSE 288.5  R2 0.510
Classifier: LightGBM Classifier  F1 0.778  AUC 0.921
Fill rate : 79.9% (P0 baseline) -> 93.9% (P4)
Cost      : +4.6% vs the P0 baseline
